# Fine Tuning CodeBert

In [ ]:
!pip install seqeval evaluate datasets

# Load Data

In [119]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="train.jsonl")
dataset = dataset["train"].train_test_split(test_size=0.1,shuffle=True)


In [14]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

# Load JSONL dataset
dataset = load_dataset("json", data_files="train.jsonl")
dataset = dataset["train"].train_test_split(test_size=0.1)

# Define label list
label_list = ["O", "B-CODE", "I-CODE"]
label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}

# Load tokenizer with add_prefix_space=True
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base", add_prefix_space=True)

# Tokenization and label alignment
def tokenize_and_align_labels(example):
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",  # <- ensures uniform length
        max_length=128,        # <- pick a reasonable value (adjust as needed)
    )
    word_ids = tokenized.word_ids()
    labels = example["labels"]
    aligned_labels = []

    for word_idx in word_ids:
        if word_idx is None or word_idx >= len(labels):
            aligned_labels.append(-100)
        else:
            aligned_labels.append(label_to_id[labels[word_idx]])

    tokenized["labels"] = aligned_labels
    return tokenized


tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=False)

# Load model
model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/codebert-base",
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)

# Evaluation metric
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p.predictions, p.label_ids
    preds = np.argmax(predictions, axis=-1)

    true_labels = [[id_to_label[label] for label in sent if label != -100] for sent in labels]
    true_preds = [[id_to_label[pred] for (pred, label) in zip(sent_pred, sent_label) if label != -100]
                  for sent_pred, sent_label in zip(preds, labels)]

    return metric.compute(predictions=true_preds, references=true_labels)

# Training arguments
training_args = TrainingArguments(
    output_dir="./codebert-code-chunker",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_total_limit=2,
)
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    data_collator=data_collator,  # <- Add this line!
)




Map:   0%|          | 0/238 [00:00<?, ? examples/s]

Map:   0%|          | 0/27 [00:00<?, ? examples/s]

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-14-b9128562822c>:82: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Train and Save

In [118]:
# Train
trainer.train()

# Save model
trainer.save_model("./codebert-code-chunker")

Epoch,Training Loss,Validation Loss,Code,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,0.050300,0.701224,"{'precision': 0.46153846153846156, 'recall': 0.32727272727272727, 'f1': 0.3829787234042553, 'number': 110}",0.461538,0.327273,0.382979,0.841393
2,0.070500,0.688804,"{'precision': 0.4634146341463415, 'recall': 0.34545454545454546, 'f1': 0.39583333333333337, 'number': 110}",0.463415,0.345455,0.395833,0.839458
3,0.032800,0.751059,"{'precision': 0.4418604651162791, 'recall': 0.34545454545454546, 'f1': 0.3877551020408163, 'number': 110}",0.441860,0.345455,0.387755,0.835590
4,0.026800,0.816193,"{'precision': 0.4852941176470588, 'recall': 0.3, 'f1': 0.37078651685393255, 'number': 110}",0.485294,0.300000,0.370787,0.839458
5,0.025900,0.846551,"{'precision': 0.4782608695652174, 'recall': 0.3, 'f1': 0.36871508379888274, 'number': 110}",0.478261,0.300000,0.368715,0.839458
6,0.024500,0.787212,"{'precision': 0.4936708860759494, 'recall': 0.35454545454545455, 'f1': 0.4126984126984127, 'number': 110}",0.493671,0.354545,0.412698,0.852998
7,0.016500,0.820735,"{'precision': 0.52, 'recall': 0.35454545454545455, 'f1': 0.42162162162162165, 'number': 110}",0.520000,0.354545,0.421622,0.854932
8,0.015000,0.833712,"{'precision': 0.4935064935064935, 'recall': 0.34545454545454546, 'f1': 0.40641711229946526, 'number': 110}",0.493506,0.345455,0.406417,0.847195
9,0.013400,0.889924,"{'precision': 0.5217391304347826, 'recall': 0.32727272727272727, 'f1': 0.40223463687150834, 'number': 110}",0.521739,0.327273,0.402235,0.845261
10,0.017000,0.872746,"{'precision': 0.49333333333333335, 'recall': 0.33636363636363636, 'f1': 0.4, 'number': 110}",0.493333,0.336364,0.400000,0.849130


Trainer is attempting to log a value of "{'precision': 0.46153846153846156, 'recall': 0.32727272727272727, 'f1': 0.3829787234042553, 'number': 110}" of type <class 'dict'> for key "eval/CODE" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'precision': 0.4634146341463415, 'recall': 0.34545454545454546, 'f1': 0.39583333333333337, 'number': 110}" of type <class 'dict'> for key "eval/CODE" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'precision': 0.4418604651162791, 'recall': 0.34545454545454546, 'f1': 0.3877551020408163, 'number': 110}" of type <class 'dict'> for key "eval/CODE" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'precision': 0.4852941176470588, 'recall': 0.3, 'f1': 0.

# Inference

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
import nltk
import re
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')


# Load tokenizer and model
model_path = "./codebert-code-chunker"
tokenizer = AutoTokenizer.from_pretrained(model_path, add_prefix_space=True)
model = AutoModelForTokenClassification.from_pretrained(model_path)
model.eval().to("cuda" if torch.cuda.is_available() else "cpu")

id2label = model.config.id2label

def annotate_code_spans(text: str) -> str:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Tokenize with NLTK
    tokens = word_tokenize(text)

    # Encode tokens
    encoded = tokenizer(tokens, is_split_into_words=True, return_tensors="pt",
                        truncation=True, padding="max_length", max_length=512)

    word_ids = encoded.word_ids(batch_index=0)
    encoded = {k: v.to(device) for k, v in encoded.items()}

    # Predict
    with torch.no_grad():
        output = model(**encoded)

    pred_ids = torch.argmax(output.logits, dim=-1).squeeze().tolist()

    # Align predictions to original words
    predicted_labels = []
    last_word_idx = None
    for i, word_idx in enumerate(word_ids):
        if word_idx is None or word_idx == last_word_idx:
            continue
        predicted_labels.append(id2label[pred_ids[i]])
        last_word_idx = word_idx

    # Rebuild annotated text with brackets
    output = []
    in_code = False

    for token, label in zip(tokens, predicted_labels):
        if label == "B-CODE":
            if in_code:
                output.append(r"\end{lstlisting}")
            output.append(r"\begin{lstlisting}")
            output.append(token)
            in_code = True
        elif label == "I-CODE":
            output.append(token)
        else:
            if in_code:
                output.append(r"\end{lstlisting}")
                in_code = False
            output.append(token)

    if in_code:
        output.append(r"\begin{lstlisting}")

    # Fix spacing: remove space before punctuation
    joined = " ".join(output)
    # joined = re.sub(r"\s+([.,!?;:%})\]])", r"\1", joined)
    # joined = re.sub(r"([\[({])\s+", r"\1", joined)

    return joined


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


# Sample

In [ ]:
text = r'''
subfigures. The command for inserting images for LATEX and PDFLaTeX can be generalized. The package used to insert images in LaTeX/PDFLaTeX is the graphicx package. Figures can be inserted via the normal figure environment as shown in the below example:

\begin{figure}[<placement-specifier>]
\centering
\includegraphics{<eps-file>}
\caption{<figure-caption>}\label{<figure-label>}
\end{figure}

In case of double column layout, the above format puts figure captions/images to single column width. To get spanned images, we need to provide \begin{figure\*} ... \end{figure\*}.

 '''

output = annotate_code_spans(text)
output = output.replace("\\end{lstlisting} \\begin{lstlisting}","")


In [123]:
output

'subfigures . The command for inserting images for LATEX and PDFLaTeX can be generalized . The package used to insert images in LaTeX/PDFLaTeX is the graphicx package . Figures can be inserted via the normal figure environment as shown in the below example : <! \\begin { figure } [ < placement-specifier > ]  \\centering  \\includegraphics { < eps-file > }  \\caption { < figure-caption > }  \\label { < figure-label > } \\end { figure } !> In case of double column layout , the above format puts figure captions/images to single column width . To get spanned images , we need to provide <! \\begin { figure\\ * } ... \\end { figure\\ * } !> .'

# Download Model

In [110]:
!zip -r codebert-code-chunker.zip /content/codebert-code-chunker
from google.colab import files
files.download("codebert-code-chunker.zip")


updating: content/codebert-code-chunker/ (stored 0%)
updating: content/codebert-code-chunker/special_tokens_map.json (deflated 84%)
updating: content/codebert-code-chunker/vocab.json (deflated 59%)
updating: content/codebert-code-chunker/merges.txt (deflated 53%)
updating: content/codebert-code-chunker/training_args.bin (deflated 51%)
updating: content/codebert-code-chunker/tokenizer_config.json (deflated 75%)
updating: content/codebert-code-chunker/tokenizer.json (deflated 82%)
updating: content/codebert-code-chunker/config.json (deflated 51%)
updating: content/codebert-code-chunker/checkpoint-150/ (stored 0%)
updating: content/codebert-code-chunker/checkpoint-150/special_tokens_map.json (deflated 84%)
updating: content/codebert-code-chunker/checkpoint-150/vocab.json (deflated 59%)
updating: content/codebert-code-chunker/checkpoint-150/merges.txt (deflated 53%)
updating: content/codebert-code-chunker/checkpoint-150/training_args.bin (deflated 51%)
updating: content/codebert-code-chunk

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>